# LexiNet Live Model Simulator

Use the trained Kneser-Ney n-gram models to play a Hangman-style word game against any secret word you choose. The word does not need to come from the train or test set; the simulator only needs lowercase alphabetic characters after normalization.

Edit `SECRET_WORD` and `MAX_FAILED_ATTEMPTS`, then run the notebook from top to bottom.

In [1]:
from pathlib import Path
from html import escape
import pickle
import random
import sys
import time

from IPython.display import HTML, clear_output, display


def find_project_root(start=None):
    """Walk upward until the LexiNet project root is found."""
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "player_agent.py").exists() and (candidate / "results" / "models").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root containing src/player_agent.py and results/models.")


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
MODEL_DIR = PROJECT_ROOT / "results" / "models"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from player_agent import GreedyPlayer


display(HTML("""
<style>
  .lexi-hero {
    border: 1px solid #d8e2ef;
    border-radius: 8px;
    padding: 18px 20px;
    margin: 8px 0 18px;
    background: linear-gradient(135deg, #f6fbff 0%, #fff8eb 100%);
    color: #162033;
    font-family: ui-sans-serif, system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
  }
  .lexi-hero h2 { margin: 0 0 6px; font-size: 23px; letter-spacing: 0; }
  .lexi-hero p { margin: 0; color: #526176; }
  .lexi-panel {
    border: 1px solid #d9e0ea;
    border-radius: 8px;
    padding: 16px;
    margin: 12px 0;
    background: #ffffff;
    color: #182235;
    font-family: ui-sans-serif, system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
    box-shadow: 0 8px 24px rgba(22, 32, 51, 0.06);
  }
  .lexi-grid { display: grid; grid-template-columns: repeat(4, minmax(0, 1fr)); gap: 10px; margin: 12px 0; }
  .lexi-stat { border: 1px solid #e3e8ef; border-radius: 8px; padding: 10px 12px; background: #f8fafc; }
  .lexi-label { font-size: 12px; text-transform: uppercase; color: #64748b; letter-spacing: 0.04em; }
  .lexi-value { font-size: 20px; font-weight: 700; margin-top: 2px; }
  .lexi-word { display: flex; flex-wrap: wrap; gap: 7px; margin: 18px 0 12px; }
  .lexi-slot {
    width: 34px;
    height: 42px;
    border: 1px solid #cfd8e5;
    border-bottom: 3px solid #8ea0b8;
    border-radius: 6px;
    display: inline-flex;
    align-items: center;
    justify-content: center;
    background: #f9fbfd;
    color: #111827;
    font-size: 22px;
    font-weight: 750;
    line-height: 1;
  }
  .lexi-slot.known { border-color: #8fc9b6; border-bottom-color: #1c8f6a; background: #effaf5; }
  .lexi-life-row { display: flex; gap: 5px; align-items: center; margin: 8px 0 12px; }
  .lexi-life {
    width: 30px;
    height: 10px;
    border-radius: 999px;
    background: #d8f3e7;
    border: 1px solid #b8decf;
  }
  .lexi-life.used { background: #ffd7d7; border-color: #f0a5a5; }
  .lexi-chip {
    display: inline-flex;
    align-items: center;
    justify-content: center;
    min-width: 28px;
    height: 28px;
    padding: 0 8px;
    margin: 3px 4px 3px 0;
    border-radius: 999px;
    border: 1px solid #cbd5e1;
    background: #f8fafc;
    font-weight: 700;
  }
  .lexi-chip.hit { color: #106b4d; background: #eaf8f1; border-color: #9cd8c0; }
  .lexi-chip.miss { color: #a23b3b; background: #fff0f0; border-color: #f3b5b5; }
  .lexi-status { font-size: 18px; font-weight: 750; margin: 6px 0; }
  .lexi-muted { color: #64748b; }
  .lexi-history { width: 100%; border-collapse: collapse; margin-top: 12px; font-size: 14px; }
  .lexi-history th, .lexi-history td { border-bottom: 1px solid #e6ebf2; padding: 8px 6px; text-align: left; }
  .lexi-history th { color: #64748b; font-size: 12px; text-transform: uppercase; letter-spacing: 0.04em; }
  .lexi-history .hit { color: #106b4d; font-weight: 700; }
  .lexi-history .miss { color: #a23b3b; font-weight: 700; }
  @media (max-width: 760px) {
    .lexi-grid { grid-template-columns: repeat(2, minmax(0, 1fr)); }
    .lexi-slot { width: 30px; height: 38px; font-size: 20px; }
  }
</style>
<div class='lexi-hero'>
  <h2>LexiNet model arena</h2>
  <p>The model sees blanks and prior guesses only. Each turn it chooses one letter using the trained n-gram language models.</p>
</div>
"""))

In [2]:
REQUIRED_N_VALUES = (3, 4, 5, 6)


def load_kneser_ney_models(model_dir=MODEL_DIR, n_values=REQUIRED_N_VALUES):
    models = {}
    missing = []
    for n in n_values:
        model_path = model_dir / f"n_{n}_gram_model_kneser_ney.pkl"
        if not model_path.exists():
            missing.append(model_path)
            continue
        with model_path.open("rb") as file:
            models[n] = pickle.load(file)
    if missing:
        missing_text = "\n".join(str(path) for path in missing)
        raise FileNotFoundError(f"Missing trained model file(s):\n{missing_text}")
    return models


def choose_n_for_word_length(word_length):
    """Match the repository's simulator behavior while keeping long words safe."""
    if word_length <= 2:
        return 3
    if word_length == 3:
        return 4
    return 6


ngram_models = load_kneser_ney_models()
word_length_to_n = {length: choose_n_for_word_length(length) for length in range(1, 201)}
PLAYER_METHOD_NAME = "best"
player = GreedyPlayer(word_length_to_n, ngram_models, PLAYER_METHOD_NAME)

loaded_models = ", ".join(f"n={n}" for n in sorted(ngram_models))
display(HTML(f"""
<div class='lexi-panel'>
  <div class='lexi-status'>Models loaded</div>
  <div class='lexi-muted'>{escape(loaded_models)} from <code>{escape(str(MODEL_DIR))}</code></div>
</div>
"""))

## Generate New Model Words

Give the model a target length and let it fabricate new word-like strings from its learned probability tables. This is probabilistic sampling rather than dictionary lookup, so rerunning the cell can produce different outputs.

In [3]:
GENERATED_WORD_LENGTH = 5
NUM_GENERATED_WORDS = 12

# Lower values make the sampler more conservative; higher values make it more adventurous.
GENERATION_TEMPERATURE = 0.85

# Options: 'best', 'bidirectional', 'forward', or 'backward'. 'best' follows the player default.
GENERATION_MODE = PLAYER_METHOD_NAME

# Keep None to use the same word-length heuristic as the game simulator.
GENERATION_N = None

# Keep None for fresh samples every run; set an integer for repeatable output.
GENERATION_SEED = None


In [3]:
def normalize_generation_mode(mode):
    mode = (mode or "best").lower()
    if mode == "best":
        return "bidirectional"
    if mode not in {"bidirectional", "forward", "backward"}:
        raise ValueError("mode must be one of: best, bidirectional, forward, backward")
    return mode


def weighted_choice(weighted_items, temperature=1.0, rng=None):
    if not weighted_items:
        return None
    rng = rng or random
    if temperature <= 0:
        return max(weighted_items, key=lambda item: item[-1])

    max_score = max(item[-1] for item in weighted_items)
    if max_score <= 0:
        return rng.choice(weighted_items)

    total = 0.0
    cumulative = []
    scale = 1.0 / max(temperature, 1e-9)
    for item in weighted_items:
        score = max(item[-1], 0.0)
        weight = (score / max_score) ** scale if score > 0 else 0.0
        total += weight
        cumulative.append((total, item))

    if total <= 0:
        return rng.choice(weighted_items)

    threshold = rng.random() * total
    for running_total, item in cumulative:
        if running_total >= threshold:
            return item
    return cumulative[-1][1]


def render_generated_word_slots(word):
    slots = []
    for letter in word:
        slots.append(f"<span class='lexi-slot known'>{escape(letter.upper())}</span>")
    return "".join(slots)


def fallback_generation_scores(current_word):
    unigram_counts = ngram_models[min(ngram_models)]["unigrams"]
    blank_positions = [index for index, ch in enumerate(current_word) if ch == "_"]
    return [
        (position, letter, float(unigram_counts.get(letter, 1)))
        for position in blank_positions
        for letter in player.alphabet
    ]


def generation_candidate_scores(current_word, n, mode="best"):
    mode = normalize_generation_mode(mode)
    probability_config = player.probability_config_for_word_length(len(current_word))
    padded_word = ["<s>"] * (n - 1) + list(current_word) + ["</s>"] * (n - 1)
    candidates = []

    for word_index, existing_letter in enumerate(current_word):
        if existing_letter != "_":
            continue
        padded_index = word_index + n - 1
        for letter in player.alphabet:
            forward_prob = 1.0
            reverse_prob = 1.0

            if mode in {"bidirectional", "forward"}:
                prefix_fwd = tuple(padded_word[padded_index - (n - 1):padded_index])
                forward_prob = player.calculate_forward_probability(prefix_fwd, letter, n, **probability_config)

            if mode in {"bidirectional", "backward"}:
                suffix_rev = tuple(padded_word[padded_index + 1:padded_index + n])
                reverse_prob = player.calculate_backward_probability(suffix_rev, letter, n, **probability_config)

            score = forward_prob * reverse_prob
            if mode == "forward":
                score = forward_prob
            elif mode == "backward":
                score = reverse_prob

            if score > 0:
                candidates.append((word_index, letter, score))

    return candidates


def generate_model_word(word_length, model_order=None, mode="best", temperature=1.0, rng=None):
    if word_length < 1:
        raise ValueError("word_length must be at least 1.")
    rng = rng or random.Random()
    n_used = model_order or choose_n_for_word_length(word_length)
    mode = normalize_generation_mode(mode)
    if n_used not in ngram_models:
        raise ValueError(f"No loaded model for n={n_used}. Available: {sorted(ngram_models)}")

    current_word = ["_"] * word_length
    trace = []
    for step in range(word_length):
        current_pattern = "".join(current_word)
        candidates = generation_candidate_scores(current_pattern, n_used, mode=mode)
        used_fallback = False
        if not candidates:
            candidates = fallback_generation_scores(current_pattern)
            used_fallback = True

        position, letter, score = weighted_choice(candidates, temperature=temperature, rng=rng)
        current_word[position] = letter
        trace.append({
            "step": step + 1,
            "position": position + 1,
            "letter": letter,
            "score": score,
            "pattern": "".join(current_word),
            "used_fallback": used_fallback,
        })

    return {
        "word": "".join(current_word),
        "word_length": word_length,
        "n_used": n_used,
        "mode": mode,
        "temperature": temperature,
        "trace": trace,
    }


def generate_model_words(word_length, num_words=10, model_order=None, mode="best", temperature=1.0, seed=None):
    rng = random.Random(seed) if seed is not None else random.Random()
    return [
        generate_model_word(
            word_length,
            model_order=model_order,
            mode=mode,
            temperature=temperature,
            rng=rng,
        )
        for _ in range(num_words)
    ]


def render_generated_words(generated_words):
    if not generated_words:
        return "<div class='lexi-panel'><div class='lexi-status'>No generated words yet.</div></div>"

    first = generated_words[0]
    words_html = []
    for item in generated_words:
        slots = render_generated_word_slots(item["word"])
        words_html.append(
            "<div style='border: 1px solid #e3e8ef; border-radius: 8px; padding: 10px 12px; background: #f8fafc;'>"
            f"<div class='lexi-word' style='margin: 0;'>{slots}</div>"
            f"<div class='lexi-muted' style='margin-top: 8px;'>sample: {escape(item['word'])}</div>"
            "</div>"
        )

    return f"""
    <div class='lexi-panel'>
      <div class='lexi-status'>Generated {len(generated_words)} model words</div>
      <div class='lexi-grid'>
        <div class='lexi-stat'><div class='lexi-label'>Word length</div><div class='lexi-value'>{first['word_length']}</div></div>
        <div class='lexi-stat'><div class='lexi-label'>Model order</div><div class='lexi-value'>n={first['n_used']}</div></div>
        <div class='lexi-stat'><div class='lexi-label'>Mode</div><div class='lexi-value' style='font-size: 16px;'>{escape(first['mode'])}</div></div>
        <div class='lexi-stat'><div class='lexi-label'>Temperature</div><div class='lexi-value'>{first['temperature']}</div></div>
      </div>
      <div style='display: grid; grid-template-columns: repeat(auto-fit, minmax(190px, 1fr)); gap: 10px;'>
        {''.join(words_html)}
      </div>
    </div>
    """


In [5]:
generated_words = generate_model_words(
    GENERATED_WORD_LENGTH,
    num_words=NUM_GENERATED_WORDS,
    model_order=GENERATION_N,
    mode=GENERATION_MODE,
    temperature=GENERATION_TEMPERATURE,
    seed=GENERATION_SEED,
)

display(HTML(render_generated_words(generated_words)))

In [60]:
GENERATED_WORD_LENGTH = 9
NUM_GENERATED_WORDS = 1

# Lower values make the sampler more conservative; higher values make it more adventurous.
GENERATION_TEMPERATURE = 0.90

# Options: 'best', 'bidirectional', 'forward', or 'backward'. 'best' follows the player default.
GENERATION_MODE = PLAYER_METHOD_NAME

# Keep None to use the same word-length heuristic as the game simulator.
GENERATION_N = None

# Keep None for fresh samples every run; set an integer for repeatable output.
GENERATION_SEED = None


generated_words = generate_model_words(
    GENERATED_WORD_LENGTH,
    num_words=NUM_GENERATED_WORDS,
    model_order=GENERATION_N,
    mode=GENERATION_MODE,
    temperature=GENERATION_TEMPERATURE,
    seed=GENERATION_SEED,
)

display(HTML(render_generated_words(generated_words)))

## Choose The Game

Set the target word and the number of allowed failed attempts. The simulator normalizes to lowercase `a-z`, so `Machine-Learning` becomes `machinelearning`.

In [61]:
SECRET_WORD = "chicanery"
MAX_FAILED_ATTEMPTS = 6

# Set ANIMATE to False when you want the final result immediately.
ANIMATE = True
DELAY_SECONDS = 0.55

# The model is mostly deterministic, but this keeps fallback guesses reproducible.
RANDOM_SEED = 7

In [62]:
def normalize_secret_word(secret_word):
    raw_word = str(secret_word).strip().lower()
    normalized = "".join(ch for ch in raw_word if "a" <= ch <= "z")
    if not normalized:
        raise ValueError("Please provide a word with at least one letter from a-z.")
    return normalized, raw_word != normalized


def reveal_letters(secret_word, visible_word, guessed_letter):
    visible = list(visible_word)
    for index, letter in enumerate(secret_word):
        if letter == guessed_letter:
            visible[index] = guessed_letter
    return "".join(visible)


def render_word_slots(visible_word):
    slots = []
    for letter in visible_word:
        slot_class = "lexi-slot known" if letter != "_" else "lexi-slot"
        slot_text = escape(letter.upper()) if letter != "_" else "&nbsp;"
        slots.append(f"<span class='{slot_class}'>{slot_text}</span>")
    return "".join(slots)


def render_life_bar(failed_count, max_failed_attempts):
    segments = []
    for index in range(max_failed_attempts):
        class_name = "lexi-life used" if index < failed_count else "lexi-life"
        segments.append(f"<span class='{class_name}'></span>")
    return "".join(segments)


def render_guess_chips(letters, class_name):
    if not letters:
        return "<span class='lexi-muted'>None yet</span>"
    return "".join(f"<span class='lexi-chip {class_name}'>{escape(letter.upper())}</span>" for letter in letters)


def render_history_table(history):
    if not history:
        return "<div class='lexi-muted'>No guesses yet.</div>"
    rows = []
    for turn in history:
        result_class = "hit" if turn["hit"] else "miss"
        result_text = "Hit" if turn["hit"] else "Miss"
        rows.append(
            "<tr>"
            f"<td>{turn['turn']}</td>"
            f"<td><strong>{escape(turn['guess'].upper())}</strong></td>"
            f"<td class='{result_class}'>{result_text}</td>"
            f"<td>{escape(turn['visible_word'].replace('_', '·').upper())}</td>"
            f"<td>{turn['misses']}</td>"
            "</tr>"
        )
    return (
        "<table class='lexi-history'>"
        "<thead><tr><th>Turn</th><th>Guess</th><th>Result</th><th>Board</th><th>Misses</th></tr></thead>"
        f"<tbody>{''.join(rows)}</tbody></table>"
    )


def render_game_html(secret_word, visible_word, history, max_failed_attempts, n_used, normalized_note=False):
    wrong_guesses = [turn["guess"] for turn in history if not turn["hit"]]
    correct_guesses = [turn["guess"] for turn in history if turn["hit"]]
    failed_count = len(wrong_guesses)
    remaining = max_failed_attempts - failed_count
    won = visible_word == secret_word
    lost = failed_count >= max_failed_attempts and not won
    if won:
        status = f"Solved in {len(history)} guesses. The secret word was {escape(secret_word.upper())}."
    elif lost:
        status = f"Game over after {failed_count} failed attempts. The secret word was {escape(secret_word.upper())}."
    else:
        status = "Model is still searching."
    note_html = ""
    if normalized_note:
        note_html = "<div class='lexi-muted'>Input was normalized to lowercase a-z before play.</div>"
    return f"""
    <div class='lexi-panel'>
      <div class='lexi-status'>{status}</div>
      {note_html}
      <div class='lexi-grid'>
        <div class='lexi-stat'><div class='lexi-label'>Word length</div><div class='lexi-value'>{len(secret_word)}</div></div>
        <div class='lexi-stat'><div class='lexi-label'>Model order</div><div class='lexi-value'>n={n_used}</div></div>
        <div class='lexi-stat'><div class='lexi-label'>Failed attempts</div><div class='lexi-value'>{failed_count}/{max_failed_attempts}</div></div>
        <div class='lexi-stat'><div class='lexi-label'>Remaining</div><div class='lexi-value'>{remaining}</div></div>
      </div>
      <div class='lexi-word'>{render_word_slots(visible_word)}</div>
      <div class='lexi-life-row'>{render_life_bar(failed_count, max_failed_attempts)}</div>
      <div><span class='lexi-label'>Correct guesses</span><br>{render_guess_chips(correct_guesses, 'hit')}</div>
      <div style='margin-top: 8px;'><span class='lexi-label'>Failed guesses</span><br>{render_guess_chips(wrong_guesses, 'miss')}</div>
      {render_history_table(history)}
    </div>
    """


def play_secret_word(secret_word, max_failed_attempts=6, animate=True, delay_seconds=0.5, random_seed=7, show=True):
    if max_failed_attempts < 1:
        raise ValueError("max_failed_attempts must be at least 1.")
    if random_seed is not None:
        random.seed(random_seed)

    secret_word, normalized_note = normalize_secret_word(secret_word)
    n_used = choose_n_for_word_length(len(secret_word))
    player.word_length_to_n[len(secret_word)] = n_used

    visible_word = "_" * len(secret_word)
    guessed_letters = set()
    history = []

    def show_board():
        if show:
            clear_output(wait=True)
            display(HTML(render_game_html(secret_word, visible_word, history, max_failed_attempts, n_used, normalized_note)))

    show_board()
    if animate and show:
        time.sleep(delay_seconds)

    while visible_word != secret_word and len([turn for turn in history if not turn["hit"]]) < max_failed_attempts:
        guessed_letter = player.guess_letter(visible_word, guessed_letters)
        guessed_letters.add(guessed_letter)
        hit = guessed_letter in secret_word
        if hit:
            visible_word = reveal_letters(secret_word, visible_word, guessed_letter)
        misses = len([turn for turn in history if not turn["hit"]]) + (0 if hit else 1)
        history.append({
            "turn": len(history) + 1,
            "guess": guessed_letter,
            "hit": hit,
            "visible_word": visible_word,
            "misses": misses,
        })
        show_board()
        if animate and show and visible_word != secret_word:
            time.sleep(delay_seconds)

    return {
        "secret_word": secret_word,
        "won": visible_word == secret_word,
        "visible_word": visible_word,
        "turns": len(history),
        "failed_attempts": len([turn for turn in history if not turn["hit"]]),
        "max_failed_attempts": max_failed_attempts,
        "n_used": n_used,
        "history": history,
    }

## Run One Game

The model starts with a fully blank word, guesses one letter per turn, and loses only when the failed-attempt counter reaches your limit.

In [69]:
game = play_secret_word(
    'CHICANERY',
    max_failed_attempts=8,#MAX_FAILED_ATTEMPTS,
    animate=ANIMATE,
    delay_seconds=DELAY_SECONDS,
    random_seed=RANDOM_SEED,
)

Turn,Guess,Result,Board,Misses
1,E,Hit,······E··,0
2,S,Miss,······E··,1
3,R,Hit,······ER·,1
4,A,Hit,····A·ER·,1
5,T,Miss,····A·ER·,2
6,L,Miss,····A·ER·,3
7,N,Hit,····ANER·,3
8,O,Miss,····ANER·,4
9,P,Miss,····ANER·,5
10,I,Hit,··I·ANER·,5


In [67]:
game = play_secret_word(
    'ANIMAX',
    max_failed_attempts=8,#MAX_FAILED_ATTEMPTS,
    animate=ANIMATE,
    delay_seconds=DELAY_SECONDS,
    random_seed=RANDOM_SEED,
)

Turn,Guess,Result,Board,Misses
1,E,Miss,······,1
2,I,Hit,··I···,1
3,S,Miss,··I···,2
4,N,Hit,·NI···,2
5,A,Hit,ANI·A·,2
6,M,Hit,ANIMA·,2
7,T,Miss,ANIMA·,3
8,L,Miss,ANIMA·,4
9,D,Miss,ANIMA·,5
10,G,Miss,ANIMA·,6


In [68]:
game = play_secret_word(
    'FLUMMOXED',
    max_failed_attempts=MAX_FAILED_ATTEMPTS,
    animate=ANIMATE,
    delay_seconds=DELAY_SECONDS,
    random_seed=RANDOM_SEED,
)

Turn,Guess,Result,Board,Misses
1,E,Hit,·······E·,0
2,S,Miss,·······E·,1
3,R,Miss,·······E·,2
4,A,Miss,·······E·,3
5,N,Miss,·······E·,4
6,D,Hit,·······ED,4
7,I,Miss,·······ED,5
8,O,Hit,·····O·ED,5
9,T,Miss,·····O·ED,6


In [ ]:
game = play_secret_word(
    SECRET_WORD,
    max_failed_attempts=MAX_FAILED_ATTEMPTS,
    animate=ANIMATE,
    delay_seconds=DELAY_SECONDS,
    random_seed=RANDOM_SEED,
)

## Try A Few More

Use this small batch view when you want quick comparisons without animation.

In [ ]:
WORDS_TO_TRY = ["simulation", "training", "network", "xylophone"]

batch_results = [
    play_secret_word(word, max_failed_attempts=MAX_FAILED_ATTEMPTS, animate=False, show=False)
    for word in WORDS_TO_TRY
]

rows = []
for result in batch_results:
    outcome = "Win" if result["won"] else "Loss"
    rows.append(
        "<tr>"
        f"<td>{escape(result['secret_word'])}</td>"
        f"<td>{outcome}</td>"
        f"<td>{result['turns']}</td>"
        f"<td>{result['failed_attempts']}/{result['max_failed_attempts']}</td>"
        f"<td>n={result['n_used']}</td>"
        "</tr>"
    )

display(HTML(
    "<div class='lexi-panel'>"
    "<div class='lexi-status'>Batch results</div>"
    "<table class='lexi-history'>"
    "<thead><tr><th>Word</th><th>Outcome</th><th>Guesses</th><th>Misses</th><th>Model</th></tr></thead>"
    f"<tbody>{''.join(rows)}</tbody>"
    "</table></div>"
))